In [5]:
import pandas as pd

# Alignement comptable propre pour Jupyter
pd.options.display.float_format = '{:,.2f} €'.format

# =============================================================================
# PARAMÈTRES GÉNÉRAUX DE LA GAMME PERMANENTE
# =============================================================================
VOLUME_BRASSIN_FINAL_LITRES = 2000.0
VOLUME_BRASSIN_PRE_EBULLITION_HL = 26.0
TAILLE_FUT_LITRES = 30.0

# =============================================================================
# ⚙️ COÛTS OPÉRATIONNELS TECHNIQUES & FLUIDES (PAR FÛT)
# =============================================================================
PRIX_EAU_M3_INDUSTRIEL = 4.00           
RATIO_LITRES_EAU_PAR_LITRE_BIERE = 5.5  
COUT_EAU_PAR_HL_FINAL = (PRIX_EAU_M3_INDUSTRIEL / 10.0) * RATIO_LITRES_EAU_PAR_LITRE_BIERE  

COUT_GAZ_CHAUFFE_PAR_HL_PREBOIL = 4.50  
COUT_ELEC_FROID_PAR_HL_PREBOIL = 1.50   
COUT_ENERGIE_PAR_HL_PREBOIL = COUT_GAZ_CHAUFFE_PAR_HL_PREBOIL + COUT_ELEC_FROID_PAR_HL_PREBOIL

# Coûts de nettoyage internes (les fûts inox étant réutilisables)
PRIX_KG_CO2_INDUSTRIEL = 4.00           
CONSOMMATION_CO2_KG_PAR_FUT = 0.35      
COUT_CHIMIE_ET_VAPEUR_PAR_FUT = 0.60    
COUT_LAVAGE_PAR_FUT = (CONSOMMATION_CO2_KG_PAR_FUT * PRIX_KG_CO2_INDUSTRIEL) + COUT_CHIMIE_ET_VAPEUR_PAR_FUT

# Droits d'accises douanières françaises
TAXE_ACCISES_PAR_HL_PAR_DEGRE = 3.91
ABV_OATMEAL_STOUT = 5.4

# Charges fixes de production (Loyer usine + Salaires brasseurs)
SALAIRE_CHARGE_BRASSEUR = 3500.0
NB_BRASSEURS = 2.0
LOYER_MENSUEL_TOTAL = 4000.0
PART_LOYER_PRODUCTION = 0.50
TOTAL_BRASSINS_MOIS = 10  # Base de répartition de la structure fixe

CHARGES_FIXES_PROD_MENSUEL = (NB_BRASSEURS * SALAIRE_CHARGE_BRASSEUR) + (LOYER_MENSUEL_TOTAL * PART_LOYER_PRODUCTION)
QUOTE_PART_FIXE_PAR_BRASSIN = CHARGES_FIXES_PROD_MENSUEL / TOTAL_BRASSINS_MOIS

# =============================================================================
# 🌾 FICHE RECETTE TECHNIQUE : OATMEAL STOUT
# =============================================================================
recette_stout_ingredients = {
    "Malts de base (Pale/Maris Otter)": {"quantite_kg": 400.0, "prix_unitaire_kg_ht": 1.10},
    "Malts Spéciaux (Chocolat/Black)":  {"quantite_kg": 80.0,  "prix_unitaire_kg_ht": 1.60},
    "Flocons d'Avoine (Oats)":          {"quantite_kg": 70.0,  "prix_unitaire_kg_ht": 1.25},
    "Houblons (East Kent Goldings)":    {"quantite_kg": 4.0,   "prix_unitaire_kg_ht": 28.00},
    "Levure & Nutriments":              {"quantite_kg": 1.0,   "prix_unitaire_kg_ht": 26.25}
}

COUT_MATIERES_PREMIERES_TOTAL = sum(item["quantite_kg"] * item["prix_unitaire_kg_ht"] for item in recette_stout_ingredients.values())
RENDEMENT_FUTS = int(VOLUME_BRASSIN_FINAL_LITRES / TAILLE_FUT_LITRES)

# =============================================================================
# CALCUL ANALYTIQUE DU COÛT DE PRODUCTION (COGS)
# =============================================================================
# 1. Fluides et Énergie
cout_eau_brassin = (VOLUME_BRASSIN_FINAL_LITRES / 100.0) * COUT_EAU_PAR_HL_FINAL
cout_energie_brassin = VOLUME_BRASSIN_PRE_EBULLITION_HL * COUT_ENERGIE_PAR_HL_PREBOIL
cout_lavage_total_brassin = RENDEMENT_FUTS * COUT_LAVAGE_PAR_FUT

# 2. Droits d'accises (20 hL commercialisés à 5.4% ABV)
hl_sortants = VOLUME_BRASSIN_FINAL_LITRES / 100.0
cout_accises_brassin = hl_sortants * ABV_OATMEAL_STOUT * TAXE_ACCISES_PAR_HL_PAR_DEGRE

# 3. Consolidation du coût du brassin
cogs_brassin_stout = (COUT_MATIERES_PREMIERES_TOTAL + cout_eau_brassin + 
                      cout_energie_brassin + cout_lavage_total_brassin + 
                      cout_accises_brassin + QUOTE_PART_FIXE_PAR_BRASSIN)

cogs_unitaire_fut = cogs_brassin_stout / RENDEMENT_FUTS

# =============================================================================
# VENTILATION DES RÉSULTATS POUR JUPYTER
# =============================================================================
# =============================================================================
# VENTILATION DES RÉSULTATS POUR JUPYTER
# =============================================================================
deconstruction_couts = {
    "Composantes du Coût": [
        "Matières Premières (Malts, Avoine, Houblon, Levure)",
        "Packaging & Conditionnement (Fûts inox réutilisables)",
        "Énergie, Eau & Fluides (Lavage CIP, CO2, Cuisson)",
        "Droits d'accises douanières (5.4% ABV)",
        "Charges Fixes d'Usine (Loyer & Salaires brasseurs)"
    ],
    "Pour 1 Brassin (20 hL)": [
        COUT_MATIERES_PREMIERES_TOTAL,
        0.00,  # Coût de matière sèche direct nul en format fût réutilisable
        cout_eau_brassin + cout_energie_brassin + cout_lavage_total_brassin,
        cout_accises_brassin,
        QUOTE_PART_FIXE_PAR_BRASSIN
    ],
    "Par Fût (30 L)": [
        COUT_MATIERES_PREMIERES_TOTAL / RENDEMENT_FUTS,
        0.00,
        (cout_eau_brassin + cout_energie_brassin + cout_lavage_total_brassin) / RENDEMENT_FUTS,
        cout_accises_brassin / RENDEMENT_FUTS,
        QUOTE_PART_FIXE_PAR_BRASSIN / RENDEMENT_FUTS
    ]
}

df_cogs_stout = pd.DataFrame(deconstruction_couts)

print("=====================================================================")
print(f"ANALYSE DE PRODUCTION : OATMEAL STOUT ({ABV_OATMEAL_STOUT}% ABV)")
print(f"Rendement industriel standard : {RENDEMENT_FUTS} fûts de 30L")
print("=====================================================================")
display(df_cogs_stout)

print("\n=====================================================================")
print(f"COÛT DE PRODUCTION GLOBAL (COGS) DU BRASSIN : {cogs_brassin_stout:,.2f} €")
print(f"COÛT DE REVIENT UNITAIRE FINAL PAR FÛT      : {cogs_unitaire_fut:,.2f} € HT")
print("=====================================================================")

ANALYSE DE PRODUCTION : OATMEAL STOUT (5.4% ABV)
Rendement industriel standard : 66 fûts de 30L


,Composantes du Coût,Pour 1 Brassin (20 hL),Par Fût (30 L)
0,"Matières Premières (Malts, Avoine, Houblon, Le...",793.75 €,12.03 €
1,Packaging & Conditionnement (Fûts inox réutili...,0.00 €,0.00 €
2,"Énergie, Eau & Fluides (Lavage CIP, CO2, Cuisson)",332.00 €,5.03 €
3,Droits d'accises douanières (5.4% ABV),422.28 €,6.40 €
4,Charges Fixes d'Usine (Loyer & Salaires brasse...,900.00 €,13.64 €



COÛT DE PRODUCTION GLOBAL (COGS) DU BRASSIN : 2,448.03 €
COÛT DE REVIENT UNITAIRE FINAL PAR FÛT      : 37.09 € HT


In [6]:
import pandas as pd

# Alignement comptable propre pour Jupyter
pd.options.display.float_format = '{:,.2f} €'.format

# =============================================================================
# PARAMÈTRES GÉNÉRAUX DE LA GAMME ÉPHÉMÈRE
# =============================================================================
VOLUME_BRASSIN_EN_CUVE_LITRES = 2000.0
VOLUME_BRASSIN_PRE_EBULLITION_HL = 26.0
TAILLE_CANETTE_LITRES = 0.44

# Structure de la distribution de la gamme éphémère (Mix des ventes)
PART_VOLUME_TAPROOM = 0.25   # Vente directe
PART_VOLUME_FRICHE = 0.75    # Distribution réseau
PART_VOLUME_DEHORS = 0.0     

# Formatage des prix de vente HT de la Double IPA (44 cL)
PRIX_CANETTE_TAPROOM_HT = 4.17   # Équivalent ~5.00€ TTC au comptoir
PRIX_CANETTE_FRICHE_HT = 2.50    # Tarif pro réseau interne

# =============================================================================
# ⚙️ LOGISTIQUE ENCANNING, FLUIDES & CHARGES FIXES
# =============================================================================
# Matière sèche (Boîtes alu, couvercles, sleeves étiquettes, cartons)
COUT_MATIERE_SECHE_PAR_CANETTE = 0.45  
# Prestation technique du prestataire mobile au volume rempli
PRESTA_ENCANNAGE_MOBILE_PAR_CANETTE = 0.33  
COUT_PACKAGING_TOTAL_CANETTE = COUT_MATIERE_SECHE_PAR_CANETTE + PRESTA_ENCANNAGE_MOBILE_PAR_CANETTE

# Énergie & Fluides (Électricité, Gaz de cuisson, Eau, CO2 de purge anti-oxygène)
PRIX_EAU_M3_INDUSTRIEL = 4.00           
RATIO_LITRES_EAU_PAR_LITRE_BIERE = 5.5  
COUT_EAU_PAR_HL_EN_CUVE = (PRIX_EAU_M3_INDUSTRIEL / 10.0) * RATIO_LITRES_EAU_PAR_LITRE_BIERE  
COUT_GAZ_CHAUFFE_PAR_HL_PREBOIL = 4.50  
COUT_ELEC_FROID_PAR_HL_PREBOIL = 1.50   
COUT_ENERGIE_PAR_HL_PREBOIL = COUT_GAZ_CHAUFFE_PAR_HL_PREBOIL + COUT_ELEC_FROID_PAR_HL_PREBOIL

# Taxes (Accises françaises pour une bière à 6.5% ABV)
TAXE_ACCISES_PAR_HL_PAR_DEGRE = 3.91
ABV_DOUBLE_IPA = 6.5

# Charges fixes de production (Loyer zone usine + 2 salaires brasseurs)
SALAIRE_CHARGE_BRASSEUR = 3500.0
NB_BRASSEURS = 2.0
LOYER_MENSUEL_TOTAL = 4000.0
PART_LOYER_PRODUCTION = 0.50
TOTAL_BRASSINS_MOIS = 10  # Clé de répartition mensuelle

CHARGES_FIXES_PROD_MENSUEL = (NB_BRASSEURS * SALAIRE_CHARGE_BRASSEUR) + (LOYER_MENSUEL_TOTAL * PART_LOYER_PRODUCTION)
QUOTE_PART_FIXE_PAR_BRASSIN = CHARGES_FIXES_PROD_MENSUEL / TOTAL_BRASSINS_MOIS

# =============================================================================
# 🌾 FICHE RECETTE TECHNIQUE : DOUBLE IPA DDH (12 g/L)
# =============================================================================
# Taux de chargement en grains élevé pour atteindre 6.5% ABV
# Dry-hop lourd de 24 kg au total sur le fermenteur (12 g/L sur 2000 L)
recette_dipa_ingredients = {
    "Malt de base (Pilsner)":           {"quantite_kg": 520.0, "prix_unitaire_kg_ht": 1.15},
    "Malt Spécial (Cara50)":            {"quantite_kg": 60.0,  "prix_unitaire_kg_ht": 1.45},
    "Houblons Amérisants (Ébullition)": {"quantite_kg": 3.5,   "prix_unitaire_kg_ht": 24.00},
    "Houblons Aromatiques (DDH 12g/L)": {"quantite_kg": 24.0,  "prix_unitaire_kg_ht": 32.00},
    "Levure US-05 & Nutriments":        {"quantite_kg": 1.2,   "prix_unitaire_kg_ht": 28.00}
}

COUT_MATIERES_PREMIERES_TOTAL = sum(item["quantite_kg"] * item["prix_unitaire_kg_ht"] for item in recette_dipa_ingredients.values())

# Impact des 12 g/L de houblon en pellet : absorption de 10,5L de bière par kg de houblon introduit
PERTE_DRYHOP_LITRES = recette_dipa_ingredients["Houblons Aromatiques (DDH 12g/L)"]["quantite_kg"] * 10.5
VOLUME_NET_COMMERCIALISE_LITRES = VOLUME_BRASSIN_EN_CUVE_LITRES - PERTE_DRYHOP_LITRES
RENDEMENT_CANETTES = int(VOLUME_NET_COMMERCIALISE_LITRES / TAILLE_CANETTE_LITRES)

# =============================================================================
# CALCUL ANALYTIQUE DU COÛT DE PRODUCTION (COGS)
# =============================================================================
# 1. Coûts variables (appliqués sur le volume mis en œuvre)
cout_eau_brassin = (VOLUME_BRASSIN_EN_CUVE_LITRES / 100.0) * COUT_EAU_PAR_HL_EN_CUVE
cout_energie_brassin = VOLUME_BRASSIN_PRE_EBULLITION_HL * COUT_ENERGIE_PAR_HL_PREBOIL

# 2. Coûts variables (appliqués sur les volumes nets commercialisés/conditionnés)
hl_sortants = VOLUME_NET_COMMERCIALISE_LITRES / 100.0
cout_accises_brassin = hl_sortants * ABV_DOUBLE_IPA * TAXE_ACCISES_PAR_HL_PAR_DEGRE
cout_packaging_brassin = RENDEMENT_CANETTES * COUT_PACKAGING_TOTAL_CANETTE

# 3. Consolidation du coût du brassin
cogs_brassin_dipa = (COUT_MATIERES_PREMIERES_TOTAL + cout_eau_brassin + 
                     cout_energie_brassin + cout_accises_brassin + 
                     cout_packaging_brassin + QUOTE_PART_FIXE_PAR_BRASSIN)

cogs_unitaire_canette = cogs_brassin_dipa / RENDEMENT_CANETTES

# =============================================================================
# VENTILATION DES RÉSULTATS POUR JUPYTER
# =============================================================================
deconstruction_couts = {
    "Composantes du Coût": [
        "Matières Premières (Malt Pilsner, Cara50, DDH)",
        "Packaging (Matière sèche + Ligne mobile)",
        "Énergie, Eau & Fluides de process",
        "Droits d'accises douanières (6.5% ABV)",
        "Charges Fixes d'Usine (Loyer & Salaires)"
    ],
    "Pour 1 Brassin (20 hL)": [
        COUT_MATIERES_PREMIERES_TOTAL,
        cout_packaging_brassin,
        cout_eau_brassin + cout_energie_brassin,
        cout_accises_brassin,
        QUOTE_PART_FIXE_PAR_BRASSIN
    ],
    "Par Canette (44 cL)": [
        COUT_MATIERES_PREMIERES_TOTAL / RENDEMENT_CANETTES,
        cout_packaging_brassin / RENDEMENT_CANETTES,
        (cout_eau_brassin + cout_energie_brassin) / RENDEMENT_CANETTES,
        cout_accises_brassin / RENDEMENT_CANETTES,
        QUOTE_PART_FIXE_PAR_BRASSIN / RENDEMENT_CANETTES
    ]
}

df_cogs_dipa = pd.DataFrame(deconstruction_couts)

print("=====================================================================")
print(f"ANALYSE DE PRODUCTION : DOUBLE IPA DDH ({ABV_DOUBLE_IPA}% ABV)")
print(f"Rendement net après perte de sédimentation du DDH : {RENDEMENT_CANETTES} canettes")
print("=====================================================================")
display(df_cogs_dipa)

print("\n=====================================================================")
print(f"COÛT DE PRODUCTION GLOBAL (COGS) DU BRASSIN : {cogs_brassin_dipa:,.2f} €")
print(f"COÛT DE REVIENT UNITAIRE FINAL CIBLE        : {cogs_unitaire_canette:,.2f} € HT")
print("=====================================================================")

ANALYSE DE PRODUCTION : DOUBLE IPA DDH (6.5% ABV)
Rendement net après perte de sédimentation du DDH : 3972 canettes


,Composantes du Coût,Pour 1 Brassin (20 hL),Par Canette (44 cL)
0,"Matières Premières (Malt Pilsner, Cara50, DDH)","1,570.60 €",0.40 €
1,Packaging (Matière sèche + Ligne mobile),"3,098.16 €",0.78 €
2,"Énergie, Eau & Fluides de process",200.00 €,0.05 €
3,Droits d'accises douanières (6.5% ABV),444.25 €,0.11 €
4,Charges Fixes d'Usine (Loyer & Salaires),900.00 €,0.23 €



COÛT DE PRODUCTION GLOBAL (COGS) DU BRASSIN : 6,213.01 €
COÛT DE REVIENT UNITAIRE FINAL CIBLE        : 1.56 € HT
